# Text2Cypher Fine-Tuning: Resource-Constrained Implementation

**Note on Hardware Limitations:** This notebook was specifically engineered to execute within the strict compute and memory constraints of a free-tier Google Colab T4 GPU. 

To prevent Out-Of-Memory (OOM) crashes and fit within execution time limits, deliberate compromises were made:
* **Downsampled Dataset:** Only a tiny fraction of the available data was used.
* **Minimal Epochs:** The model was trained for the absolute minimum time (1 epoch).
* **Forced Packing:** Sequence packing was enabled without Flash Attention to speed up processing, which is known to cause cross-contamination and hallucinations.

**If you are running this on an A100, L4, or a local rig with more VRAM, please read the inline `[OPTIMAL CONFIG]` comments throughout the code to disable these limitations and achieve high-accuracy Cypher generation.**

In [15]:
!pip install -q -U \
    transformers \
    datasets \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    sentencepiece

In [16]:
!pip install -q accelerate

### 1. Verify Environment and Dependencies

In [17]:
import torch
import transformers
import peft
import trl
import bitsandbytes
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("bf16 there:",torch.cuda.is_bf16_supported())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi

PyTorch: 2.10.0+cu128
Transformers: 5.17.0
PEFT: 0.21.0
TRL: 1.13.0
CUDA Available: True
bf16 there: True
GPU: Tesla T4
Fri Sep 18 07:40:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             31W /   70W |     105MiB /  15360MiB |      0%      Default |
|                   

### 2. Download and Sample the Dataset

In [29]:
from datasets import load_dataset

from huggingface_hub import login
import os

os.environ['HF_TOKEN'] = ''
login(token=os.environ.get("HF_TOKEN"))
# Load the official Neo4j text2cypher dataset
dataset = load_dataset("neo4j/text2cypher-2024v1")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# [OPTIMAL CONFIG]: If you have the compute time, train on the full dataset.
# The T4 constraint forced us to shrink this to 3000 train / 300 test examples to finish in a reasonable time.
# To do it properly, remove the `.select(range(3000))` and `.select(range(300))` calls.
train_data = dataset["train"].shuffle(seed=42).select(range(3000))
test_data = dataset["test"].shuffle(seed=42).select(range(300))

print("Train size:", len(train_data))
print("Test size:", len(test_data))

Train size: 3000
Test size: 300


### 3. Convert Dataset into Instruction Format
We format our inputs using ChatML-style system, user, and assistant tokens.

In [30]:
def format_example(example):
    return {
        "text": f"""<|im_start|>system
You are an expert Neo4j Cypher query generator.

Given a graph schema and a natural language question, generate only the valid Cypher query.
Do not explain the query.
<|im_end|>
<|im_start|>user
Schema:
{example["schema"]}

Question:
{example["question"]}
<|im_end|>
<|im_start|>assistant
{example["cypher"]}
<|im_end|>"""
    }

train_data = train_data.map(format_example)
test_data = test_data.map(format_example)
print(train_data[0]["text"])

<|im_start|>system
You are an expert Neo4j Cypher query generator.

Given a graph schema and a natural language question, generate only the valid Cypher query.
Do not explain the query.
<|im_end|>
<|im_start|>user
Schema:
Node properties:
- **Movie**
  - `title`: STRING Example: "The Matrix"
  - `votes`: INTEGER Min: 1, Max: 5259
  - `tagline`: STRING Example: "Welcome to the Real World"
  - `released`: INTEGER Min: 1975, Max: 2012
- **Person**
  - `born`: INTEGER Min: 1929, Max: 1996
  - `name`: STRING Example: "Keanu Reeves"
Relationship properties:
- **ACTED_IN**
  - `roles: LIST` Min Size: 1, Max Size: 6
- **REVIEWED**
  - `summary: STRING` Available options: ['Pretty funny at times', 'A solid romp', 'Silly, but fun', 'You had me at Jerry', 'An amazing journey', 'Slapstick redeemed only by the Robin Williams and ', 'Dark, but compelling', 'The coolest football movie ever', 'Fun, but a little far fetched']
  - `rating: INTEGER` Min: 45, Max:  100
The relationships:
(:Person)-[:ACTED

Loading the model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},     # was "auto" — stop sharding a 1B model across 2 GPUs
    dtype=torch.bfloat16    # torch_dtype also works but dtype is the current arg name
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

### 5. Evaluate the Base Model (Before Fine-tuning)
Let's test the baseline capabilities of the non-fine-tuned model.

In [32]:
schema_test = """
Node labels:
Vehicle:
- make: STRING
- model: STRING
- year: INTEGER
Problem:
- name: STRING
Relationships:
(Vehicle)-[:HAS_PROBLEM]->(Problem)
"""
question_test = "Which Toyota vehicles have overheating problems?"

prompt = f"""<|im_start|>system
You are an expert Neo4j Cypher query generator.

Generate only valid Cypher.
Do not explain the query.
<|im_end|>
<|im_start|>user
Schema:
{schema_test}

Question:
{question_test}
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )
generated = outputs[0][inputs["input_ids"].shape[-1]:]
print("=== BASE MODEL RESPONSE ===")
print(tokenizer.decode(generated, skip_special_tokens=True))

=== BASE MODEL RESPONSE ===
MATCH (v:Vehicle { make: 'Toyota' })-[:HAS_PROBLEM]->(p:Problem)
WHERE p.name = 'Overheating'
RETURN v.name AS VehicleName, p.name AS ProblemName
<|im_end|>
<|im_start|>assistant
MATCH (v:Vehicle { make: 'Toyota' })-[:HAS_PROBLEM]->(p:Problem)
WHERE p.name = 'Overheating'
RETURN v.name AS VehicleName, p.name AS ProblemName
<|im_end|>
<|im_start|>assistant
MATCH (v:Vehicle { make: 'Toyota


### Define Expected Cypher Query (Ground Truth)
Here we explicitly define the target query (`expected_cypher`) and reference the base model's generated query (`base_query`) so we can compute similarity metrics.

In [33]:
expected_cypher = """
MATCH (v:Vehicle)-[:HAS_PROBLEM]->(p:Problem {name: 'Overheating'})
WHERE v.make = 'Toyota'
RETURN v
"""

# Dynamically capture the decoded base model output instead of using a hardcoded string
base_query = tokenizer.decode(generated, skip_special_tokens=True).strip()

print("=== EXPECTED CYPHER ===")
print(expected_cypher.strip())
print("\n=== BASE MODEL GENERATED CYPHER ===")
print(base_query)

=== EXPECTED CYPHER ===
MATCH (v:Vehicle)-[:HAS_PROBLEM]->(p:Problem {name: 'Overheating'})
WHERE v.make = 'Toyota'
RETURN v

=== BASE MODEL GENERATED CYPHER ===
MATCH (v:Vehicle { make: 'Toyota' })-[:HAS_PROBLEM]->(p:Problem)
WHERE p.name = 'Overheating'
RETURN v.name AS VehicleName, p.name AS ProblemName
<|im_end|>
<|im_start|>assistant
MATCH (v:Vehicle { make: 'Toyota' })-[:HAS_PROBLEM]->(p:Problem)
WHERE p.name = 'Overheating'
RETURN v.name AS VehicleName, p.name AS ProblemName
<|im_end|>
<|im_start|>assistant
MATCH (v:Vehicle { make: 'Toyota


### Programmatic Similarity Scoring
Let's define a simple metric function to evaluate the exact overlap (F1 Score) of tokens and sequence similarity between the generated and expected Cypher queries.

In [34]:
from collections import Counter
import difflib

def compute_metrics(generated, expected):
    # Normalize whitespace and lowercase
    gen_tokens = [t for t in generated.lower().split() if t.strip()]
    exp_tokens = [t for t in expected.lower().split() if t.strip()]

    # 1. Token-level F1 Score
    gen_counter = Counter(gen_tokens)
    exp_counter = Counter(exp_tokens)

    intersection = sum((gen_counter & exp_counter).values())
    precision = intersection / len(gen_tokens) if gen_tokens else 0.0
    recall = intersection / len(exp_tokens) if exp_tokens else 0.0

    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    # 2. Sequence Matcher ratio (Gestalt Pattern Matching)
    seq_ratio = difflib.SequenceMatcher(None, " ".join(gen_tokens), " ".join(exp_tokens)).ratio()

    return {
        "Token Precision": precision,
        "Token Recall": recall,
        "Token F1-Score": f1,
        "Sequence Similarity": seq_ratio
    }

base_metrics = compute_metrics(base_query, expected_cypher)
print("=== BASE MODEL METRIC SCORES ===")
for metric, score in base_metrics.items():
    print(f"{metric}: {score:.4f}")

=== BASE MODEL METRIC SCORES ===
Token Precision: 0.1163
Token Recall: 0.5000
Token F1-Score: 0.1887
Sequence Similarity: 0.3714


### 6. Prepare the Model for QLoRA Training

In [35]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [36]:
model.print_trainable_parameters()
dtypes_found = {}
for name, param in model.named_parameters():
    dtypes_found.setdefault(param.dtype, []).append(name)

for dtype, names in dtypes_found.items():
    print(dtype, len(names), "params — e.g.", names[:3])

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
torch.float32 258 params — e.g. ['base_model.model.model.embed_tokens.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight']
torch.uint8 112 params — e.g. ['base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight']


### 7. Configure SFTTrainer & Train

In [37]:
from trl import SFTConfig, SFTTrainer
# [OPTIMAL CONFIG]: 1 epoch is not enough for a 1B parameter model to master Cypher syntax.
    # Change to 3-5 epochs if you have the compute budget.
training_args = SFTConfig(
    output_dir="./text2cypher",
    num_train_epochs=1, #make it atleast 4 so that the model is fine tuned poperly 
    per_device_train_batch_size=2,    
    per_device_eval_batch_size=2,       
    gradient_accumulation_steps=4,      # was 8 — bigger batch replaces need for accumulation
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=25,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    max_length=1024,
    dataset_text_field="text",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    loss_type="nll",
    packing=False
)


In [38]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer
)

trainer.train()

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported Flash Attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported Flash Attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
108,0.274113,0.431374,0.235630,0.951957,1738237.000000


TrainOutput(global_step=108, training_loss=0.8958791935885394, metrics={'train_runtime': 10500.2694, 'train_samples_per_second': 0.164, 'train_steps_per_second': 0.01, 'total_flos': 1.0266913916940288e+16, 'train_loss': 0.8958791935885394, 'epoch': 1.0})

### 8. Save the LoRA Adapter

In [ ]:
ADAPTER_PATH = "./text2cypher-lora"
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

### 9. Evaluate and Compare the Fine-Tuned Model

In [40]:
from peft import PeftModel

# Load fresh base model and combine with adapter for evaluation
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16)

base_model_eval.config.pad_token_id = tokenizer.pad_token_id
base_model_eval.generation_config.pad_token_id = tokenizer.pad_token_id
base_model_eval.generation_config.bos_token_id = tokenizer.bos_token_id

eval_model = PeftModel.from_pretrained(base_model_eval, ADAPTER_PATH)
eval_model.eval()

inputs_eval = tokenizer(prompt, return_tensors="pt").to(eval_model.device)
with torch.no_grad():
    outputs_eval = eval_model.generate(
        **inputs_eval,
        max_new_tokens=128,
        do_sample=False
    )
generated_eval = outputs_eval[0][inputs_eval["input_ids"].shape[-1]:]
print("=== FINE-TUNED MODEL RESPONSE ===")
print(tokenizer.decode(generated_eval, skip_special_tokens=True))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

=== FINE-TUNED MODEL RESPONSE ===
MATCH (v:Vehicle) WHERE v.make = 'Toyota' AND v.year < 2000 AND v.year > 1995 AND HAS_PROBLEM {problem: 'overheating'} RETURN v.name
<|im_end|>


In [41]:
fine_tuned_query = tokenizer.decode(generated_eval, skip_special_tokens=True).strip()

fine_tuned_metrics = compute_metrics(fine_tuned_query, expected_cypher)
print("=== FINE-TUNED MODEL METRIC SCORES ===")
for metric, score in fine_tuned_metrics.items():
    print(f"{metric}: {score:.4f}")

=== FINE-TUNED MODEL METRIC SCORES ===
Token Precision: 0.2857
Token Recall: 0.6000
Token F1-Score: 0.3871
Sequence Similarity: 0.4115


In [ ]:
!zip -r text2cypher-lora.zip ./text2cypher-lora